# 06 · Clasificador de envases: entrenamiento a la vista en Jupyter

Mismo entrenamiento que `scripts/clasificador_entrenar.py`, pero **paso a paso y dentro del notebook**: se ven
los datos, la arquitectura, la barra de progreso de cada época (`model.fit(verbose=1)`), TensorBoard en vivo,
las curvas, la matriz de confusión y las predicciones. Las funciones (dataset, callbacks, métricas) son las
**mismas** del script, así que los resultados son comparables con `modelos/clasificador/comparacion.csv`.

**Cómo usarlo**
1. Kernel: `inventariovision` (la primera celda imprime el Python; debe terminar en `envs\inventariovision\python.exe`).
2. Elegir `MODELO` y `RAPIDO` en la celda de configuración y correr todo (`Run All`).
3. Para entrenar otro modelo: cambiar `MODELO`, **reiniciar el kernel** y correr todo de nuevo.

Los resultados van a `modelos/clasificador/jupyter/` (o `jupyter_rapido/`): **no** pisan los modelos que usa el
sistema (`modelos/clasificador/clasificador.keras`).

| modelo | qué es | última corrida completa (con 9 clases, antes de quitar 7Up) |
|---|---|---|
| `cnn` | `modeloCNN` de la práctica 01, con softmax | 10 épocas, ~16 min, test acc 0.870 |
| `cnn_ad` | + aumento de datos + Dropout | 40 épocas, ~73 min, test acc 0.907 |
| `mobilenet` | transfer learning MobileNetV2 + fine-tuning | 25 épocas, ~117 min, test acc 0.918 |

In [ ]:
# 0. Configuración: lo único que hace falta tocar
MODELO = 'cnn'      # 'cnn' | 'cnn_ad' | 'mobilenet'
RAPIDO = False      # True: 100 recortes por clase y 3 épocas -> se ve todo el proceso en pocos minutos (métricas sin valor)
HILOS = 4           # hilos de TensorFlow en CPU (0 = sin límite). Con 4 la PC sigue usable mientras entrena.

In [ ]:
# 1. Entorno y rutas (y límite de hilos ANTES de usar TensorFlow)
import os, sys, json, time, shutil, tempfile
from datetime import datetime
from pathlib import Path
print('Python:', sys.executable)

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO))
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
# %tensorboard lanza el tensorboard.exe que aparece primero en el PATH (el de base): se fuerza el del entorno.
os.environ['TENSORBOARD_BINARY'] = str(Path(sys.executable).parent / 'Scripts' / 'tensorboard.exe')

from src import config
import scripts.clasificador_entrenar as ce
from src.vision import clasificador_modelos as cm

ce.configurar_tf(HILOS)          # hilos + semillas (keras.utils.set_random_seed(42))
ce.mantener_despierto()          # Windows: que la PC no se suspenda mientras entrena

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf, keras
from IPython.display import display, Markdown, Image as IImage
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

CLASES = config.nombres_clases()
P = cm.PARAMETROS[MODELO]
TAMANO = P['tamano']
MAX_POR_CLASE = 100 if RAPIDO else None
EPOCAS = 3 if RAPIDO else P['epocas']
EPOCAS_FT = (2 if RAPIDO else P.get('epocas_finetune', 0)) if MODELO == 'mobilenet' else 0

DIR_RECORTES = config.DIR_DATOS / 'recortes'
SALIDA = config.DIR_MODELOS / 'clasificador' / ('jupyter_rapido' if RAPIDO else 'jupyter')
FECHA = datetime.now()
DIR_LOGS = config.DIR_LOGS / 'clasificador' / 'jupyter' / f'{MODELO}_{FECHA:%Y%m%d-%H%M%S}'
SALIDA.mkdir(parents=True, exist_ok=True)

print('TensorFlow', tf.__version__, '· Keras', keras.__version__, '· GPU:', tf.config.list_physical_devices('GPU') or 'no (CPU)')
print(f'Modelo: {MODELO} · entrada {TAMANO}px · épocas {EPOCAS}' + (f' + {EPOCAS_FT} de fine-tuning' if EPOCAS_FT else '')
      + (' · MODO RÁPIDO' if RAPIDO else ''))
print('Recortes:', DIR_RECORTES, '\nSalida  :', SALIDA, '\nLogs    :', DIR_LOGS)

## 1. Los datos

Cada imagen es un **recorte** de una caja etiquetada en los datasets públicos (lo genera el agente Datos en
`datos/recortes/<split>/<clase>/`). Los splits están separados por **foto original**: dos recortes de la misma
foto nunca quedan uno en train y otro en test (eso sería *fuga de datos*).

In [ ]:
splits = {s: ce.listar_split(DIR_RECORTES, s, MAX_POR_CLASE) for s in ce.SPLITS}
conteo = pd.DataFrame({s: np.bincount(y, minlength=len(CLASES)) for s, (_, y) in splits.items()}, index=CLASES)
conteo.loc['TOTAL'] = conteo.sum()
display(conteo)
conteo.drop('TOTAL')['train'].plot.barh(title='Recortes de entrenamiento por clase', figsize=(6, 3.5)); plt.show()

**Desbalance → `class_weight`.** Una clase con la mitad de ejemplos pesa el doble en la pérdida, para que la
red no aprenda a "nunca decir" las clases raras. Se topea en 10 para que una clase muy chica no desestabilice.

In [ ]:
PESOS_CLASE = ce.pesos_clase(splits['train'][1], len(CLASES))
pd.Series({CLASES[i]: round(w, 2) for i, w in PESOS_CLASE.items()}, name='peso').to_frame().T

### Pipeline `tf.data`

Para cada ruta: leer el JPEG → `letterbox` a un cuadrado de `TAMANO` px **sin deformar** (una botella no debe
parecer una lata) → cachear en RAM → mezclar (solo train) → lotes de 32. La normalización y el aumento de datos
van **dentro** del modelo, así el `.keras` recibe píxeles 0-255 tal cual.

In [ ]:
dir_cache = Path(tempfile.gettempdir()) / f'clasificador_cache_jupyter_{os.getpid()}_{MODELO}'
ds = {s: ce.crear_dataset(r, y, TAMANO, mezclar=(s == 'train'), archivo_cache=dir_cache / s)
      for s, (r, y) in splits.items() if len(y)}

# Un lote de train, tal como lo recibe la red
x, y = next(iter(ds['train']))
fig, ejes = plt.subplots(2, 8, figsize=(16, 4.4))
for a, im, et in zip(ejes.flat, x.numpy(), y.numpy()):
    a.imshow(im.astype('uint8')); a.set_title(CLASES[et], fontsize=8); a.axis('off')
plt.suptitle(f'Un lote de entrenamiento: {tuple(x.shape)} (lote, alto, ancho, canales)'); plt.show()

## 2. El modelo

- `cnn`: Conv32/64/128 + MaxPool → Flatten → Dense100 → **Dense9 softmax** (en la práctica 01 era 1 neurona sigmoid).
- `cnn_ad`: igual + capas de aumento (rotación ±10°, zoom, traslación, brillo, contraste) + Dropout(0.5) + Dense250.
- `mobilenet`: MobileNetV2 preentrenada en ImageNet **congelada** + cabeza nueva; después *fine-tuning* de las
  últimas 30 capas con lr 1e-5.

Pérdida `sparse_categorical_crossentropy` (etiquetas enteras 0..8) y optimizador Adam.

In [ ]:
modelo = cm.construir(MODELO, len(CLASES), pesos='imagenet', tamano=TAMANO)
modelo.summary()
print(f'Entrenables: {sum(int(np.prod(w.shape)) for w in modelo.trainable_weights):,} de {modelo.count_params():,}')

## 3. TensorBoard (en vivo)

Correr esta celda **antes** de entrenar. Se actualiza al final de cada época (botón de recarga arriba a la
derecha, o se refresca solo cada 30 s). Pestaña *SCALARS*: `epoch_loss` y `epoch_accuracy` de train y validation.
Si no se ve dentro del notebook: abrir http://localhost:6006 en el navegador.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{config.DIR_LOGS / 'clasificador' / 'jupyter'}" --port 6006

## 4. Entrenamiento

Callbacks (los mismos del script):
- **EarlyStopping** sobre `val_loss` (paciencia 5-8): corta cuando deja de mejorar y **restaura la mejor época**.
- **ModelCheckpoint**: guarda en disco el mejor modelo apenas aparece (si se corta la sesión, no se pierde).
- **ReduceLROnPlateau**: baja el *learning rate* ×0.3 cuando `val_loss` se estanca.
- **CSVLogger** (historial) y **TensorBoard**.

Cada línea de época muestra `loss`/`accuracy` de entrenamiento y `val_loss`/`val_accuracy`. Lo que importa es
**validación**: si `accuracy` sube y `val_accuracy` no, el modelo está memorizando (sobreajuste).

> Ojo: interrumpir esta celda deja el modelo a medio entrenar. Lo mejor hasta ese momento ya está en `RUTA_KERAS`.

In [ ]:
RUTA_KERAS = SALIDA / f'{MODELO}.keras'
HISTORIAL = SALIDA / f'{MODELO}_historial.csv'
HISTORIAL.unlink(missing_ok=True); RUTA_KERAS.unlink(missing_ok=True)

crono = ce.cronometro()
t0 = time.time()
hist = modelo.fit(ds['train'], validation_data=ds['val'], epochs=EPOCAS, class_weight=PESOS_CLASE, shuffle=False,
                  callbacks=[*ce.callbacks(MODELO, RUTA_KERAS, HISTORIAL, DIR_LOGS), crono], verbose=1)
if RUTA_KERAS.exists():
    modelo.load_weights(RUTA_KERAS)       # la mejor época (aunque EarlyStopping no haya cortado)
print(f'\nFase 1: {len(hist.epoch)} épocas en {(time.time() - t0) / 60:.1f} min · '
      f'mejor val_loss {min(hist.history["val_loss"]):.4f} · mejor val_accuracy {max(hist.history["val_accuracy"]):.4f}')

### 4 b. Fine-tuning (solo `mobilenet`)

Se descongelan las últimas 30 capas de la base (las BatchNormalization siguen congeladas) y se sigue entrenando
con un *learning rate* 100 veces menor, para ajustar los filtros de ImageNet a nuestros envases sin "olvidarlos".
Con `cnn` / `cnn_ad` esta celda no hace nada.

In [ ]:
crono_ft = ce.cronometro()
if MODELO == 'mobilenet':
    mejor = float(np.min(hist.history['val_loss']))
    cm.habilitar_finetune(modelo, P['capas_finetune'], P['lr_finetune'])
    print(f'Entrenables ahora: {sum(int(np.prod(w.shape)) for w in modelo.trainable_weights):,}')
    hechas = len(hist.epoch)
    hist_ft = modelo.fit(ds['train'], validation_data=ds['val'], initial_epoch=hechas, epochs=hechas + EPOCAS_FT,
                         class_weight=PESOS_CLASE, shuffle=False,
                         callbacks=[*ce.callbacks(MODELO, RUTA_KERAS, HISTORIAL, DIR_LOGS, mejor), crono_ft], verbose=1)
    modelo.load_weights(RUTA_KERAS)
else:
    print('No aplica a', MODELO)
MINUTOS = (time.time() - t0) / 60
modelo.save(RUTA_KERAS)
print(f'Total: {MINUTOS:.1f} min · guardado en {RUTA_KERAS}')

## 5. Curvas de entrenamiento vs. validación

In [ ]:
h = pd.read_csv(HISTORIAL)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, met in zip(ax, ['accuracy', 'loss']):
    a.plot(h['epoch'] + 1, h[met], 'o-', label='entrenamiento')
    a.plot(h['epoch'] + 1, h[f'val_{met}'], 'o-', label='validación')
    a.set_title(met); a.set_xlabel('época'); a.grid(alpha=.3); a.legend()
ax[2].semilogy(h['epoch'] + 1, h['learning_rate'], 'o-'); ax[2].set_title('learning rate'); ax[2].set_xlabel('época')
if MODELO == 'mobilenet' and len(hist.epoch) < len(h):
    for a in ax: a.axvline(len(hist.epoch) + 0.5, ls='--', c='gray')    # inicio del fine-tuning
plt.tight_layout(); plt.show()
print('Segundos por época:', [round(t) for t in crono.tiempos + crono_ft.tiempos])

## 6. Evaluación en validación y test

- **Precisión** de una clase: de lo que el modelo dijo "coca_lata", cuánto era coca_lata.
- **Recall**: de las coca_lata reales, cuántas encontró.
- **Matriz de confusión** (filas = real, columnas = predicha): lo que importa en el inventario son las confusiones
  **botella ↔ lata de la misma marca**.
- El test **no se usa para elegir** nada: solo para reportar.

In [ ]:
MET = {'modelo': MODELO, 'tamano': TAMANO, 'parametros': int(modelo.count_params()),
       'epocas': len(crono.tiempos) + len(crono_ft.tiempos), 'minutos': round(MINUTOS, 2),
       'seg_por_epoca': [round(t, 1) for t in crono.tiempos + crono_ft.tiempos],
       'n': {s: int(len(y)) for s, (_, y) in splits.items()},
       'class_weight': {CLASES[i]: round(w, 3) for i, w in PESOS_CLASE.items()},
       'prueba': RAPIDO, 'max_por_clase': MAX_POR_CLASE, 'fecha': FECHA.isoformat(timespec='seconds'),
       'logs': str(DIR_LOGS), 'origen': 'notebooks/06_clasificador_jupyter.ipynb'}
for s in ('val', 'test'):
    if s in ds:
        MET[s] = ce.evaluar(modelo, ds[s], splits[s][1])
        ce.graficar_matriz(MET[s]['matriz_confusion'], CLASES, f'{MODELO} · {s}', SALIDA / f'matriz_{MODELO}_{s}.png')
        m = MET[s]
        display(Markdown(f"### {s}: accuracy **{m['accuracy']:.3f}** · loss {m['loss']:.3f} · F1 macro {m['macro_f1']:.3f}"))
        display(pd.DataFrame(m['reporte']).T.loc[CLASES, ['precision', 'recall', 'f1-score', 'support']].round(3))
        display(IImage(filename=str(SALIDA / f'matriz_{MODELO}_{s}.png'), width=560))
MET['umbral_sugerido'] = ce.sugerir_umbral(MET['val']['barrido_umbral'])
(SALIDA / f'{MODELO}_metricas.json').write_text(json.dumps(MET, indent=2, ensure_ascii=False), encoding='utf-8')
%matplotlib inline
# ↑ graficar_matriz (del script) pasa matplotlib a 'Agg' para guardar el PNG: se vuelve a mostrar dentro del notebook

In [ ]:
# Confusiones clave en test (n = cantidad de recortes de la clase "de" que el modelo llamó "a")
cc = MET['test']['confusiones_clave'] if MET.get('test', {}).get('n') else MET['val']['confusiones_clave']
print('botella↔lata:', cc['total_botella_lata'])
display(pd.DataFrame(cc['botella_lata']).sort_values('n', ascending=False))

## 7. Umbral de confianza

Si la probabilidad máxima del softmax es menor que el umbral, el sistema responde **desconocido** en vez de
arriesgar. Más umbral → se acepta menos (cobertura baja) pero lo aceptado es más confiable.

In [ ]:
b = pd.DataFrame(MET['val']['barrido_umbral'])
ax = b.plot(x='umbral', y=['cobertura', 'accuracy_aceptadas'], marker='o', figsize=(7, 3.5), grid=True)
ax.axvline(MET['umbral_sugerido'], ls='--', c='gray')
ax.set_title(f"Validación · umbral sugerido {MET['umbral_sugerido']} (accuracy ≥ {ce.PRECISION_OBJETIVO} sobre lo aceptado)")
plt.show()

## 8. Predicciones sobre recortes de test (verde = acierto, rojo = error)

In [ ]:
rng = np.random.default_rng(0)
rutas_t, y_t = splits['test']
idx = np.concatenate([rng.permutation(np.where(y_t == k)[0])[:2] for k in range(len(CLASES))])
x_t = np.stack([cm.letterbox(cm.cargar_imagen(rutas_t[i]), TAMANO) for i in idx])
probs = modelo.predict(x_t, verbose=0)
fig, ejes = plt.subplots(2, len(CLASES), figsize=(18, 5))
for a, im, i, p in zip(ejes.T.flat, x_t, idx, probs):
    pred, conf = int(p.argmax()), float(p.max())
    txt = CLASES[pred] if conf >= MET['umbral_sugerido'] else f'desconocido ({CLASES[pred]})'
    a.imshow(im); a.axis('off')
    a.set_title(f'real {CLASES[y_t[i]]}\n{txt} {conf:.2f}', fontsize=7, color='green' if pred == y_t[i] else 'red')
plt.tight_layout(); plt.show()

## 9. Comparación con los demás modelos entrenados en esta carpeta

`comparar_y_elegir` arma `comparacion.csv` con todos los `<modelo>_metricas.json` de `SALIDA` y copia el de mayor
`val_accuracy` a `SALIDA/clasificador.keras`. Entrenando los tres (cambiando `MODELO`) se reproduce la tabla
completa. Abajo, la tabla oficial (la del script) como referencia.

In [ ]:
ce.comparar_y_elegir(SALIDA)
display(pd.read_csv(SALIDA / 'comparacion.csv'))
oficial = config.DIR_MODELOS / 'clasificador' / 'comparacion.csv'
if oficial.exists():
    display(Markdown('**Referencia: `modelos/clasificador/comparacion.csv` (entrenamiento con el script)**'))
    display(pd.read_csv(oficial))

In [ ]:
# Limpieza: libera la RAM de TensorFlow y la caché de imágenes
keras.backend.clear_session()
shutil.rmtree(dir_cache, ignore_errors=True)
print('Listo. Para usar este modelo en el sistema: copiar', SALIDA / 'clasificador.keras', 'a', config.DIR_MODELOS / 'clasificador')